# 10. Paoay: screening and crop maps

Create boundary-limited carrot/cabbage/potato predictions plus Unknown and separate missing inputs. Add reversible spectral/land-cover review flags; do not hard-exclude pixels without validated evidence. The conservative Dynamic World map remains secondary context.


## 1. Setup

Finish 09 first. Keep the existing score thresholds. Screening uses the full eight-bin vegetation record; a bare date alone does not remove a field. No training or downloads.


In [ ]:
from pathlib import Path
import json
import sys
repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import maps, crop_baseline, screening
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress
settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
rules = {"minimum_crop_probability": 0.60, "minimum_crop_margin": 0.10, "minimum_supported_bins": 4}
show_preview = True
show_context_preview = False  # Optional conservative Dynamic World map, separate from the crop baseline.
prediction_pointer = paoay_root / "predictions/prediction_stage.json"
map_root = paoay_root / "maps"
map_pointer = map_root / "map_stage.json"
screening_root = paoay_root / "screening"
screening_pointer = screening_root / "screening_stage.json"
crop_pointer = paoay_root / "crop_baseline/crop_baseline_stage.json"
screening_rules = dict(screening.RULES)  # NDVI maximum < 0.20 is an illustrative FLAG, not an exclusion.
for pointer in (map_pointer, screening_pointer, crop_pointer):
    write_stage_pointer(pointer, status="pending", reason="Notebook 10 map assembly is incomplete")
stages = ["Verify prediction and guidance inputs", "Assemble attributed maps", "Assess screening flags", "Review class areas and sources", "Preview and verify maps", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("10_paoay_maps", stages, report_folder.parent / "progress", {"rules": rules})


## 2. Verify inputs

Use the exact prediction-bound imagery/guidance chain. New or failed runs leave current map and flag selections pending.


In [ ]:
pipeline_progress.begin_stage("Verify prediction and guidance inputs")
prediction_handoff = resolve_stage_pointer(prediction_pointer)
request = maps.prepare_request(prediction_handoff, map_root, rules)
baseline_request = crop_baseline.prepare_request(prediction_handoff, paoay_root / "crop_baseline", {"minimum_top_probability": rules["minimum_crop_probability"], "minimum_margin": rules["minimum_crop_margin"]})
show_table([{"rule": key.replace("_", " "), "value": value} for key, value in baseline_request["identity"]["rules"].items()], title="Crop score rules")
print("Map extent: union of KMZ reference polygons; outside pixels masked.")
print("Screening policy: flags only; hard exclusions disabled pending validation.")
pipeline_progress.end_stage()


## 3. Assemble maps

Preserve the raw scores. Mask outside reference polygons, retain holes and count overlapping polygons once in the overall map. Unknown is score rejection, not other crops.


In [ ]:
pipeline_progress.begin_stage("Assemble attributed maps")
def map_progress(items):
    return pipeline_progress.track(items, "Assemble map contexts", unit="half-month")
crop_baseline_handoff = crop_baseline.build(baseline_request)
baseline_bundle = crop_baseline.load_bundle(crop_baseline_handoff)
map_handoff = maps.build(request, iterator=map_progress)
bundle = maps.load_map_bundle(map_handoff)
pipeline_progress.end_stage()


## 4. Assess screening

Summarize all four indices over time using the existing bands. NDVI maximum below 0.20 throughout eight valid bins is an illustrative weak-vegetation flag. Missing bins remain a separate evidence flag. Repeated supported water, built, woody or bare context is flagged for review. NDRE/NDMI/SAVI support interpretation, without separate arbitrary exclusion thresholds. B5/B11 are natively 20 m despite the 10 m grid. Flags can overlap. Hard exclusions remain disabled until tested on labelled crop and noncrop examples.


In [ ]:
pipeline_progress.begin_stage("Assess screening flags")
screening_request = screening.prepare_request(crop_baseline_handoff, map_handoff, screening_root, screening_rules)
screening_handoff = screening.build(screening_request)
screening_bundle = screening.load_bundle(screening_handoff)
show_table([{**row, "pixels": int(row["pixels"]), "area_ha": float(row["area_ha"]), "boundary_percent": float(row["boundary_percent"])} for row in screening_bundle["rows"] if int(row["pixels"]) > 0], title="Screening review flags by boundary (may overlap)")
print("Full flag table:", screening_bundle["folder"] / "boundary_flags.csv")
print("NDVI, NDRE, NDMI and SAVI time summaries:", screening_bundle["folder"] / "screening.tif")
print("Flags do not remove predictions, identify other crops or establish noncrop ground truth.")
pipeline_progress.end_stage()


## 5. Review crop areas

Accepted predictions are conditional on the three trained classes. Other crops may still receive a confident target label. Nothing is automatically converted to other crops.


In [ ]:
pipeline_progress.begin_stage("Review class areas and sources")
show_table([{**r, "pixels": int(r["pixels"]), "area_ha": float(r["area_ha"]), "roi_percent": float(r["roi_percent"])} for r in baseline_bundle["summary"]], title="PRIMARY: current three-crop model + Unknown")
print("Crop-only model scores; these areas are not verified planted area.")
print("Conservative Dynamic World context tables:", bundle["folder"])
print("Other-crop training methods: TODO only.")
pipeline_progress.end_stage()


## 6. Preview and publish

Show a zoomed overview and a separate zoomed panel for each boundary. Only display limits change; raster resolution and predictions remain unchanged. Verify all three products before updating their pointers. Previous products remain in their original keyed folders.


In [ ]:
pipeline_progress.begin_stage("Preview and verify maps")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    from crop_pipeline.paoay import map_previews
    for suffix, plot in [("zoomed", map_previews.plot_overview), ("boundaries", map_previews.plot_boundaries)]:
        figure = plot(baseline_bundle)
        preview = paoay_root / "crop_baseline/previews" / (baseline_bundle["manifest"]["bundle_id"] + "_" + suffix + ".png")
        preview.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(preview, dpi=150)
        display(figure)
        plt.close(figure)
        print("Zoomed crop preview:", preview)
if show_context_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    context_figure = maps.plot_maps(bundle)
    display(context_figure)
    plt.close(context_figure)
maps.load_map_bundle(map_handoff)
crop_baseline.load_bundle(crop_baseline_handoff)
screening.load_bundle(screening_handoff)
write_stage_pointer(map_pointer, map_handoff)
write_stage_pointer(crop_pointer, crop_baseline_handoff)
write_stage_pointer(screening_pointer, screening_handoff)
pipeline_progress.end_stage()


## 7. Next step

Notebook 11 combines reference and field review. Do not treat January observations as verification of 2024 occupancy without historical evidence. Corrected maps must remain separate from frozen predictions.

TODO: pooled other-crop training and individual-crop training with grouped reporting remain deferred.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Primary crop baseline handoff:", crop_baseline_handoff)
print("Separate context map handoff:", map_handoff)
print("Current map pointer:", map_pointer)
print("Screening flag handoff:", screening_handoff)
print("Raw crop model outputs and fitted model: preserved")
print("Next: notebook 11 reviews references and freezes the preliminary map for field checks.")
print("Dynamic World attribution:", bundle["manifest"]["attribution"])
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
